Let me explore each of the four folders in detail.Let me clone the repo to see all of its files at once.Let me read a strategic sample of files to understand patterns. Start with the foundational files:Now let me look at representative scripts from each category. Starting with Metadata (the simplest, most tutorial-like):Now the Data folder:Now Administration:Now the Samples folder (advanced):Let me look at a few more to round out the picture:## tm1py-samples — repository structure

The official Cubewise repo `cubewise-code/tm1py-samples` is a learning catalogue of ~60 ready-to-run Python scripts plus a few Jupyter notebooks. Everything is built on top of TM1py, the Python wrapper around the TM1 / Planning Analytics REST API.

The repo is split into four top-level folders that map to a clear progression in skill:

```
tm1py-samples/
├── check.py            ← interactive connectivity test (run this first)
├── config.ini          ← shared credentials file used by every script
├── Administration/     ← housekeeping & operational tasks
├── Data/               ← reading/writing cell values
├── Metadata/           ← creating/updating TM1 objects
└── Samples/            ← end-to-end use cases (FX, ML, forecasting, GA)
```

## Classification of the samples

### 1. Bootstrap (root)

- `check.py` — prompts for connection params, opens a `TM1Service`, prints the server name. The "hello world" of TM1py.
- `config.ini` — `[tm1srv01]` / `[tm1srv02]` sections with address, port, user, base64-encoded password, ssl. Every other script reads this with `configparser` and unpacks it via `**config['tm1srv01']`.

### 2. Metadata — building blocks (CRUD on TM1 objects)

This folder is essentially a tour of the `TM1py.Objects` and `TM1py.Services` API. Each script is short and follows a "create instance of an Object → call the matching Service" pattern.

| Object family            | Sample scripts                                                                                                                                                                                                                                            |
| ------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Cubes                    | `cube_create.py`, `cube_get.py`, `cube_delete.py`, `cube_get_cells_number.py`                                                                                                                                                                             |
| Dimensions / Hierarchies | `dim_create.py`, `dim_get.py`, `dim_create_huge_dimension.py`, `dim_hierarchy_get_default_member.py`, `dim_hierarchy_subset_get.py`, `dim_hierarchy_subset_update.py`, `dimension_update.py`, `update_default_member.py`, `sync_dimension_and_subsets.py` |
| Subsets                  | `dim_subset_create.py`                                                                                                                                                                                                                                    |
| Views                    | `cube_view_create.py`, `cube_view_delete_all_private_mdx_views.py`, `cube_view_delete_views_subsets_through_regex.py`, `cube_view_generate_mdx_from_native_view.py`                                                                                       |
| Processes                | `process_create.py`, `process_get.py`, `process_update.py`, `process_run_code.py`, `process_run_in_parallel.py`                                                                                                                                           |
| Chores                   | `chore_create.py`, `chore_get.py`, `chore_update.py`                                                                                                                                                                                                      |
| Security                 | `user_create.py`                                                                                                                                                                                                                                          |
| Annotations              | `annotation_create_and_delete.py`                                                                                                                                                                                                                         |
| Applications             | `application_get.py` (extracts an .xlsx from the Application tree)                                                                                                                                                                                        |

### 3. Data — getting numbers in and out of cubes

- `cube_write_data.py` — basic synchronous `tm1.cubes.cells.write_values(cube, {coords: value})`.
- `cube_write_data_fast.py` — same job but parallelised: `asyncio` + `ThreadPoolExecutor` + `write_values_through_cellset` for ~1M cells.
- `cube_load_data_from_csv.py` — read ~1M-line CSV, build a `dict` of coordinates → value, push in one batch.
- `cube_load_data_from_cube.py` — copy data between cubes.
- `cube_data_into_pandas_dataframe.py` — `execute_mdx` then `Utils.build_pandas_dataframe_from_cellset` plus `df.describe()`.
- `mdx_query_different_formats.py` — the most useful single Data sample: it shows every read format TM1py supports (raw dict, case-insensitive dict, values-only, csv, dataframe, ui_array, ui_dygraph) with the same MDX, plus calculated members and 3-D queries commented out as variants.
- `cube_query_view_vs_mdx.py` — micro-benchmark: 20 runs of `execute_view` vs `execute_mdx`.
- `print_data_from_mdx.py`, `print_data_from_mdx_view.py`, `print_pivot_table_from_mdx.py` — terminal-friendly printers.
- Two notebooks (`reading_data.ipynb`, `data_structures_in_python.ipynb`) use the included `reading_data.csv` / `plan_BudgetPlan.csv`.

### 4. Administration — operating a TM1 server

These move from "manage one object" to "scan the whole model".

- **Session management**: `session_reuse.py` (share a `session_id` across `TM1Service` instances to skip CAM auth), `session_persist.py`, `session_custom_context.py` (set `session_context=` so requests show up labelled in Arc).
- **Bulk cleanup**: `cleanup_model.py` and `cube_view_delete_views_subsets_through_regex.py` walk every cube/dimension/process/chore and delete anything matching a regex like `^temp_` or `^TM1py`.
- **Inventory / health**: `dim_find_unused.py` (set difference between all dims and dims used in any cube), `groups_find_unused.py`, `cube_rules_stats.py` (sort cubes by SKIPCHECK, UNDEFVALS, # rules, # feeders), `list_tm1_users/` (user inventory across multiple servers).
- **Performance / diagnostics**: `cube_processing_feeders.py` (re-fire feeders cube-by-cube and parse `MessageLog` to time each one), `mdx_stress_test.py` and `stress_test_rest_api_calls.py` (concurrent MDX with `asyncio` + `run_in_executor`).
- **Audit**: `transactionlog_entries_since_timestamp.py`, `transactionlog_delta_requests.py` (pull the `}TransactionLog` cube programmatically).
- **Scheduling**: `chore_reschedule_all_plus_one_hour.py` (loop every chore, call `chore.reschedule(hours=-1)`, push back).

### 5. Samples — full use cases (the "why bother" folder)

- `credentials_best_practice.py` — instead of a base64 password in config.ini, look it up in the OS keyring (`keyring` lib → Windows Credential Manager / macOS Keychain).
- `samples_setup.py` — creates the TM1 cubes & dimensions the other Samples assume (`TM1py FX Rates`, `TM1py Econ`, `TM1py Stock Prices`, …).
- **External-data-into-TM1**:
  - `fx_rates_ecb_setup.py` / `fx_rates_ecb_to_cube_daily.py` — fetch the ECB XML feed, parse with `ElementTree`, write rates.
  - `fx_rates_fred_to_cube_daily.py`, `fx_rates_fred_to_cube_monthly.py`, `gdp_data_from_fred_to_cube.py` — same pattern via the FRED API.
  - `stock_prices_to_cube.py` — `quandl.get("WIKI/IBM")` → cellset → cube.
  - `GoogleAnalyticsAPI/google_analytics_to_csv.py` — pulls GA dimensions/metrics into a CSV that a TI process can then load.
- **Data-science use cases** (notebooks):
  - `Cubike/` — fictional bike-share company; uses Facebook Prophet for time-series forecasting (`time_series_forecasting.ipynb`) and Pandas/Plotly for EDA (`exploratory_analysis.ipynb`, `pandas_in_depth.ipynb`).
  - `Cubank/` — fictional bank; loan-default classification with scikit-learn against a TM1 cube of ~95 MB of LendingClub loan data.
- `samples_for_beginners.ipynb` — a guided introduction; the first cell goes straight from `TM1Service(...)` to `execute_view_dataframe_pivot` to a styled DataFrame.

## The TM1py methodology that the samples teach

Working through the repo, the same skeleton recurs:

In [ ]:
import configparser
from TM1py.Objects import <SomeObject>      # Cube, Dimension, Hierarchy, Element,
                                            # ElementAttribute, Subset, Process,
                                            # Chore, ChoreTask, ChoreFrequency,
                                            # ChoreStartTime, Annotation, User
from TM1py.Services import TM1Service

config = configparser.ConfigParser()
config.read(r'..\config.ini')

with TM1Service(**config['tm1srv01']) as tm1:
    ...

The key principles the samples reinforce:

**1. The two-layer API.** Everything you build is a `TM1py.Objects.X` (a plain Python data class — `Cube`, `Dimension`, `Hierarchy`, `Element`, `Subset`, `Process`, `Chore`, etc.). Everything you do to TM1 goes through a `TM1py.Services` collection accessed off the `tm1` handle. The collections are nested the way the model is: `tm1.cubes`, `tm1.cubes.cells`, `tm1.cubes.views`, `tm1.cubes.annotations`, `tm1.dimensions`, `tm1.dimensions.subsets`, `tm1.dimensions.hierarchies`, `tm1.processes`, `tm1.chores`, `tm1.security`, `tm1.applications`, `tm1.server`.

**2. Standard verbs.** Every service exposes the same vocabulary: `create(obj)`, `get(name)`, `get_all()`, `get_all_names()`, `update(obj)`, `delete(name)`, `exists(name)`. Once you've seen one CRUD sample (`cube_create.py`) you can read all of them.

**3. Round-tripping is the editing pattern.** `process_update.py` is the template: `p = tm1.processes.get(name)` → mutate `p` in Python (`p.add_parameter(...)`, `p.epilog_procedure = ...`) → `tm1.processes.update(p)`. Same for chores, dimensions, cubes.

**4. Always use the context manager.** `with TM1Service(...) as tm1:` guarantees the session is logged out. The exceptions in the repo (`sync_dimension_and_subsets.py`, `annotation_create_and_delete.py`) are the ones that explicitly need a long-lived or manually managed session.

**5. Cells are coordinate→value dicts.** Writing data is always: build a `dict` keyed by tuples of element names in dimension order, then one call to `tm1.cubes.cells.write_values(cube, cellset)`. This is the entire pattern in `cube_write_data.py`, `cube_load_data_from_csv.py`, `fx_rates_ecb_to_cube_daily.py`, `stock_prices_to_cube.py`, etc.

**6. Reading data has many shapes — pick the right one.** `mdx_query_different_formats.py` is the cheat sheet: `execute_mdx` (dict), `execute_mdx_values` (just the numbers), `execute_mdx_csv`, `execute_mdx_dataframe` (pandas, ready to go), `execute_mdx_raw` (full REST payload with cell properties), and the UI-shaped `execute_mdx_ui_array` / `execute_mdx_ui_dygraph` for charting front-ends.

**7. MDX is preferred over named views, but you can convert.** `cube_view_generate_mdx_from_native_view.py` shows `tm1.cubes.views.get_native_view(...).MDX` to extract an MDX string from a native view; `cube_query_view_vs_mdx.py` benchmarks the two paths.

**8. Bulk and parallel work uses asyncio + run_in_executor.** Both `process_run_in_parallel.py` and `mdx_stress_test.py` use the same recipe: define a sync function that takes `(tm1, args)`, gather futures with `loop.run_in_executor(None, fn, tm1, …)`, then `await` them. `cube_write_data_fast.py` adds a `ThreadPoolExecutor(max_workers=10)` and `write_values_through_cellset` for fast bulk writes.

**9. Whole-model tasks are just loops over `get_all_names()`.** Cleanup, audit, find-unused, sort-by-rule-count — all of them iterate `tm1.cubes.get_all_names()` (and `dimensions`, `processes`, `chores`) and apply a regex or predicate. Control objects are filtered out with `not name.startswith('}')`.

**10. The TI bridge is `processes.execute_ti_code`.** When TM1py doesn't wrap an operation directly (`SaveDataAll`, `SecurityRefresh`, `CubeProcessFeeders`, `DeleteAllPersistentFeeders`), `process_run_code.py` shows you can ship raw TI statements to the server without creating a stored process.

**11. Credentials.** The repo's own files use base64 in `config.ini` for convenience but every script carries the comment "storing the credentials in a file is not recommended … see Samples/credentials_best_practice.py". That sample uses the `keyring` library to fetch the password from the OS credential store. For real deployments you also have `session_id` reuse (`session_reuse.py`) and `session_context` labelling (`session_custom_context.py`).

**12. TM1py is a glue layer, not a destination.** The Samples folder makes the larger point: TM1py's value is connecting TM1 to the rest of the Python ecosystem — `requests` + `xml.etree` for ECB, `quandl` for stocks, `google-api-python-client` for GA, `pandas` for analysis, `prophet` for forecasting, `scikit-learn` for classification.

A reasonable learning path through the repo: `check.py` → a couple of `Metadata/*_get.py` and `*_create.py` → `Data/cube_write_data.py` and `mdx_query_different_formats.py` → `Data/cube_data_into_pandas_dataframe.py` → `Administration/cleanup_model.py` and `dim_find_unused.py` → `Samples/credentials_best_practice.py` → `Samples/fx_rates_ecb_to_cube_daily.py` → the Cubike or Cubank notebook.